# Dodgeball YOLO Training

This notebook fine-tunes an Ultralytics YOLO detection model on the dodgeball dataset.

Classes:

- `0`: ball
- `1`: player


In [ ]:
from pathlib import Path

from ultralytics import YOLO

DATASET = Path("data/dodgeball/data.yaml")
assert DATASET.exists(), f"Dataset config not found: {DATASET}"

print(DATASET.read_text())


## Check The Dataset

Before training, confirm that every image has a matching YOLO label file. The training split uses frames `000001` to `000260`, and the validation split uses frames `000261` to `000325`.


In [ ]:
for split in ("train", "val"):
    images = sorted((Path("data/dodgeball/images") / split).glob("*.jpg"))
    labels = sorted((Path("data/dodgeball/labels") / split).glob("*.txt"))
    image_names = {path.stem for path in images}
    label_names = {path.stem for path in labels}

    print(f"{split}: {len(images)} images, {len(labels)} labels")
    print("missing labels:", sorted(image_names - label_names))
    print("missing images:", sorted(label_names - image_names))


## Train The Model

This starts from pretrained YOLO weights and adapts them to your two classes. The saved run that already exists in this project used these settings:

- model: `yolo26m.pt`
- epochs: `50`
- image size: `640`
- batch size: `8`
- device: `mps`
- run name: `dodgeball-v1`


In [ ]:
model = YOLO("models/yolo26m.pt")

results = model.train(
    data=str(DATASET),
    epochs=50,
    imgsz=640,
    batch=8,
    device="mps",
    project="runs/detect",
    name="dodgeball-v1",
)


## Use The Best Model

After training, Ultralytics saves the strongest checkpoint as `best.pt`. Use that model for prediction. Your project also keeps a copy in `models/best.pt`.


In [ ]:
best_model = YOLO("models/best.pt")

prediction_results = best_model.predict(
    "data/raw/videos/dodgeball.mp4",
    save=True,
    project="runs/detect",
    name="predict",
    exist_ok=True,
)

print(prediction_results[0])


## Resume Or Continue Training

Use `resume=True` with `last.pt` only if a run was interrupted. If you add more labelled data later, usually start a new run from `models/best.pt` instead.


In [ ]:
# Resume an interrupted run:
# model = YOLO("models/last.pt")
# results = model.train(resume=True)

# Continue improving from the current best model after adding/correcting data:
# model = YOLO("models/best.pt")
# results = model.train(
#     data=str(DATASET),
#     epochs=30,
#     imgsz=640,
#     batch=8,
#     device="mps",
#     project="runs/detect",
#     name="dodgeball-v2",
# )
